In [ ]:
from pathlib import Path

import gc
import json
import os
import re
import subprocess
import sys
import time

import numpy as np
import pandas as pd


In [ ]:
# ============================================================
# Paths and fixed FRD protocol
# ============================================================

PROJECT_ROOT = Path.cwd()

EVALUATION_ROOT = (
    PROJECT_ROOT
    / "evaluation_200"
)

REAL_DIR = (
    EVALUATION_ROOT
    / "real"
)

MODEL_DIRS = {
    "ddpm_v5":
        EVALUATION_ROOT
        / "ddpm_v5",

    "conditional_ddpm_v3":
        EVALUATION_ROOT
        / "conditional_ddpm_v3",

    "conditional_ldm_v4":
        EVALUATION_ROOT
        / "conditional_ldm_v4"
}

MODEL_PREFIXES = {
    "ddpm_v5":
        "ddpm_v5",

    "conditional_ddpm_v3":
        "conditional_ddpm_v3",

    "conditional_ldm_v4":
        "conditional_ldm_v4"
}

RESULTS_DIR = (
    PROJECT_ROOT
    / "upstream_results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MAIN_RESULTS_JSON = (
    RESULTS_DIR
    / "upstream_results.json"
)

FRD_RESULTS_JSON = (
    RESULTS_DIR
    / "frd_results.json"
)

FRD_PROTOCOL_JSON = (
    RESULTS_DIR
    / "frd_evaluation_protocol.json"
)

N_VOLUMES = 200

FRD_VERSION = "v1"

FRD_NUM_WORKERS = 1

# Keep False during normal/resumed runs.
FORCE_RECOMPUTE = False


print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Python executable:",
    sys.executable
)

print(
    "FRD version:",
    FRD_VERSION
)

print(
    "FRD CPU workers:",
    FRD_NUM_WORKERS
)


In [ ]:
# ============================================================
# Verify package and all four 200-volume datasets
# ============================================================

import frd_score


print(
    "frd_score version:",
    getattr(
        frd_score,
        "__version__",
        "unknown"
    )
)


def list_nifti_files(
    directory: Path,
    prefix: str
):

    return sorted(
        directory.glob(
            f"{prefix}_*.nii.gz"
        )
    )


real_files = list_nifti_files(
    REAL_DIR,
    "real"
)


if len(real_files) != N_VOLUMES:

    raise RuntimeError(
        "Real reference set is incomplete: "
        f"{len(real_files)}/{N_VOLUMES}"
    )


dataset_counts = {
    "real":
        len(real_files)
}


for model_name, directory in (
    MODEL_DIRS.items()
):

    files = list_nifti_files(
        directory,
        MODEL_PREFIXES[
            model_name
        ]
    )

    dataset_counts[
        model_name
    ] = len(files)


    if len(files) != N_VOLUMES:

        raise RuntimeError(
            f"{model_name} is incomplete: "
            f"{len(files)}/{N_VOLUMES}"
        )


display(
    pd.DataFrame(
        [
            {
                "Dataset":
                    name,

                "NIfTI volumes":
                    count,

                "Complete":
                    count == N_VOLUMES
            }
            for name, count in (
                dataset_counts.items()
            )
        ]
    )
)


print(
    "All FRD input datasets contain "
    f"{N_VOLUMES} volumes."
)


In [ ]:
# ============================================================
# Persistent and atomic result storage
# ============================================================

def atomic_write_json(
    path: Path,
    payload
):

    temporary_path = (
        path.with_suffix(
            path.suffix
            + ".tmp"
        )
    )


    with temporary_path.open(
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            allow_nan=False
        )


    os.replace(
        temporary_path,
        path
    )


def load_json_or_default(
    path: Path,
    default
):

    if not path.exists():
        return default


    with path.open(
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


frd_results = load_json_or_default(
    FRD_RESULTS_JSON,
    {}
)


for model_name in MODEL_DIRS:

    frd_results.setdefault(
        model_name,
        {}
    )


def frd_already_complete(
    model_name
):

    if FORCE_RECOMPUTE:
        return False


    value = (
        frd_results
        .get(
            model_name,
            {}
        )
        .get(
            "FRD"
        )
    )


    if value is None:
        return False


    try:

        return bool(
            np.isfinite(
                float(value)
            )
        )

    except (
        TypeError,
        ValueError
    ):

        return False


def save_frd_results():

    atomic_write_json(
        FRD_RESULTS_JSON,
        frd_results
    )


def merge_frd_into_main_results():

    main_results = load_json_or_default(
        MAIN_RESULTS_JSON,
        {}
    )


    for model_name in MODEL_DIRS:

        main_results.setdefault(
            model_name,
            {}
        )


        value = (
            frd_results
            .get(
                model_name,
                {}
            )
            .get(
                "FRD"
            )
        )


        if value is not None:

            main_results[
                model_name
            ][
                "FRD"
            ] = float(
                value
            )


    atomic_write_json(
        MAIN_RESULTS_JSON,
        main_results
    )


save_frd_results()

print(
    "FRD result file:",
    FRD_RESULTS_JSON
)

print(
    "Main upstream result file:",
    MAIN_RESULTS_JSON
)


In [ ]:
# ============================================================
# Run one FRDv1 comparison in a fresh subprocess
# ============================================================

FRD_OUTPUT_PATTERN = re.compile(
    r"FRD\s*\(\s*v1\s*\)\s*:\s*([^\s]+)",
    flags=re.IGNORECASE
)


def run_frd_subprocess(
    real_dir: Path,
    synthetic_dir: Path
):

    command = [
        sys.executable,
        "-m",
        "frd_score",
        str(
            real_dir.resolve()
        ),
        str(
            synthetic_dir.resolve()
        ),
        "--frd_version",
        FRD_VERSION,
        "--norm_ref",
        "d1",
        "--num_workers",
        str(
            FRD_NUM_WORKERS
        )
    ]


    # Restrict hidden numerical-library threading as well.
    environment = os.environ.copy()

    environment[
        "OMP_NUM_THREADS"
    ] = "1"

    environment[
        "MKL_NUM_THREADS"
    ] = "1"

    environment[
        "OPENBLAS_NUM_THREADS"
    ] = "1"

    environment[
        "NUMEXPR_NUM_THREADS"
    ] = "1"


    print(
        "Running:",
        " ".join(
            command
        )
    )


    started = time.perf_counter()


    process = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        env=environment,
        capture_output=True,
        text=True,
        check=False
    )


    elapsed_seconds = (
        time.perf_counter()
        - started
    )


    combined_output = (
        process.stdout
        + "\n"
        + process.stderr
    )


    if process.returncode != 0:

        raise RuntimeError(
            "FRD subprocess failed with return code "
            f"{process.returncode}.\n\n"
            f"STDOUT:\n{process.stdout[-5000:]}\n\n"
            f"STDERR:\n{process.stderr[-5000:]}"
        )


    matches = FRD_OUTPUT_PATTERN.findall(
        combined_output
    )


    if not matches:

        raise RuntimeError(
            "The FRD subprocess completed, but its "
            "numeric result could not be parsed.\n\n"
            f"Output tail:\n{combined_output[-5000:]}"
        )


    frd_value = float(
        matches[-1]
    )


    if not np.isfinite(
        frd_value
    ):

        raise RuntimeError(
            "FRD returned a non-finite value: "
            f"{frd_value}"
        )


    return (
        frd_value,
        elapsed_seconds
    )


In [ ]:
# ============================================================
# Compute FRD for the three models, saving after each one
# ============================================================

for model_name, synthetic_dir in (
    MODEL_DIRS.items()
):

    if frd_already_complete(
        model_name
    ):

        print(
            f"{model_name}: FRD already saved -> skipped"
        )

        continue


    print()
    print(
        "=" * 72
    )

    print(
        "Computing FRDv1:",
        model_name
    )

    print(
        "=" * 72
    )


    frd_value, elapsed_seconds = (
        run_frd_subprocess(
            REAL_DIR,
            synthetic_dir
        )
    )


    frd_results[
        model_name
    ] = {
        "FRD":
            float(
                frd_value
            ),

        "frd_version":
            FRD_VERSION,

        "norm_ref":
            "d1",

        "num_workers":
            FRD_NUM_WORKERS,

        "real_volumes":
            N_VOLUMES,

        "synthetic_volumes":
            N_VOLUMES,

        "elapsed_seconds":
            float(
                elapsed_seconds
            ),

        "elapsed_hours":
            float(
                elapsed_seconds
                / 3600.0
            )
    }


    # Save the dedicated FRD record first.
    save_frd_results()

    # Then merge only the FRD value into the main result JSON.
    merge_frd_into_main_results()


    print(
        model_name,
        "FRD:",
        frd_value
    )

    print(
        "Elapsed:",
        f"{elapsed_seconds / 3600.0:.2f} h"
    )

    print(
        "Results saved immediately."
    )


    gc.collect()


print()
print(
    "FRD evaluation loop finished."
)


In [ ]:
# ============================================================
# Final FRD and complete upstream tables
# ============================================================

frd_rows = []


for model_name in MODEL_DIRS:

    model_frd = (
        frd_results
        .get(
            model_name,
            {}
        )
    )


    frd_rows.append({
        "Model":
            model_name,

        "FRD":
            model_frd.get(
                "FRD",
                np.nan
            ),

        "FRD version":
            model_frd.get(
                "frd_version",
                FRD_VERSION
            ),

        "Workers":
            model_frd.get(
                "num_workers",
                FRD_NUM_WORKERS
            ),

        "Elapsed hours":
            model_frd.get(
                "elapsed_hours",
                np.nan
            )
    })


frd_df = pd.DataFrame(
    frd_rows
)


frd_csv_path = (
    RESULTS_DIR
    / "frd_comparison.csv"
)


frd_df.to_csv(
    frd_csv_path,
    index=False
)


display(
    frd_df
)


main_results = load_json_or_default(
    MAIN_RESULTS_JSON,
    {}
)


MAIN_METRICS = [
    "FID",
    "KID",
    "sFID",
    "Precision",
    "Recall",
    "Density",
    "Coverage",
    "FRD",
    "RadFID",
    "MedFID",
    "Vendi",
    "AuthPct",
    "ASW"
]


comparison_rows = []


for model_name in MODEL_DIRS:

    row = {
        "Model":
            model_name
    }


    model_results = (
        main_results.get(
            model_name,
            {}
        )
    )


    for metric in MAIN_METRICS:

        row[
            metric
        ] = model_results.get(
            metric,
            np.nan
        )


    comparison_rows.append(
        row
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


comparison_path = (
    RESULTS_DIR
    / "upstream_comparison.csv"
)


comparison_df.to_csv(
    comparison_path,
    index=False
)


display(
    comparison_df
)


completion_rows = []


for model_name in MODEL_DIRS:

    model_results = (
        main_results.get(
            model_name,
            {}
        )
    )


    missing_metrics = [
        metric
        for metric in MAIN_METRICS
        if (
            metric not in model_results
            or not np.isfinite(
                float(
                    model_results[
                        metric
                    ]
                )
            )
        )
    ]


    completion_rows.append({
        "Model":
            model_name,

        "Complete":
            len(
                missing_metrics
            ) == 0,

        "Missing metrics":
            ", ".join(
                missing_metrics
            )
    })


completion_df = pd.DataFrame(
    completion_rows
)


completion_path = (
    RESULTS_DIR
    / "upstream_metric_completeness.csv"
)


completion_df.to_csv(
    completion_path,
    index=False
)


display(
    completion_df
)


print(
    "FRD comparison:",
    frd_csv_path
)

print(
    "Full upstream comparison:",
    comparison_path
)

print(
    "Completeness audit:",
    completion_path
)


In [ ]:
# ============================================================
# Save the FRD evaluation protocol
# ============================================================

frd_protocol = {
    "metric":
        "Fréchet Radiomic Distance",

    "version":
        FRD_VERSION,

    "reference":
        "Real 200 preprocessed BraTS 2023 T2-FLAIR volumes",

    "synthetic_models":
        list(
            MODEL_DIRS.keys()
        ),

    "volumes_per_distribution":
        N_VOLUMES,

    "normalization_reference":
        "d1",

    "cpu_workers":
        FRD_NUM_WORKERS,

    "execution_isolation":
        "one fresh Python subprocess per model",

    "image_types":
        [
            "Original",
            "LoG",
            "Wavelet"
        ],

    "force_recompute":
        FORCE_RECOMPUTE
}


atomic_write_json(
    FRD_PROTOCOL_JSON,
    frd_protocol
)


print(
    "FRD protocol saved:",
    FRD_PROTOCOL_JSON
)
